# LifeMate trên Colab

Chuẩn bị một lần:
1. **Runtime → Change runtime type → GPU A100** (fine-tune ~1 giờ; L4 chạy được nhưng ~2–2.5 giờ). T4 có thể không chạy được vLLM bản mới.
2. Upload `train+public_test.zip` (có sẵn trong folder project trên Mac) lên Google Drive tại `MyDrive/zalo_lifemate/train+public_test.zip`.
3. Tạo GitHub token (fine-grained, quyền *Contents: Read* cho repo `FirstKhoi/zalo_lifemate`), thêm vào **Colab Secrets** (biểu tượng chìa khoá bên trái) với tên `GH_TOKEN` và bật *Notebook access*.

Nếu sau cell cài đặt mà một dòng `!...` báo `SyntaxError`: **Thời gian chạy → Khởi động lại phiên**, chạy lại cell 1 rồi đi tiếp (không cần cài lại).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + data

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
assert os.path.isfile(f"{DRIVE}/train+public_test.zip"), f"chưa có {DRIVE}/train+public_test.zip trên Drive"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!ls train+public_test

In [ ]:
!pip install -q vllm peft

In [ ]:
!python test_score.py && python test_agent.py

## 2. Fine-tune LoRA trên splits/train (1223 mẫu, 2 epoch)
Adapter được lưu lên Drive sau mỗi epoch (`MyDrive/zalo_lifemate/lora`), model đã merge để ở `checkpoints/merged` cho vLLM.

In [ ]:
!python train_sft.py --out "{DRIVE}/lora" --merge-to checkpoints/merged 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|Error|Traceback"

In [ ]:
# Chỉ chạy nếu phiên bị ngắt SAU khi train xong (checkpoints/merged mất): merge lại từ adapter trên Drive
!python -c "import torch; from peft import AutoPeftModelForCausalLM as M; from transformers import AutoTokenizer as T; m = M.from_pretrained('{DRIVE}/lora', dtype=torch.bfloat16).merge_and_unload(); m.save_pretrained('checkpoints/merged'); T.from_pretrained('Qwen/Qwen3-4B-Instruct-2507').save_pretrained('checkpoints/merged')"

## 3. Đo lại trên dev bằng model đã fine-tune

In [ ]:
for split in ["dev_seen", "dev_unseen"]:
    !python agent.py --model checkpoints/merged --ids splits/{split}.txt --out runs/sft_{split}.jsonl 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|Error|Traceback"
    !python score.py runs/sft_{split}.jsonl

In [ ]:
!mkdir -p "{DRIVE}/runs" && cp runs/*.jsonl "{DRIVE}/runs/" && ls "{DRIVE}/runs"

## (Tuỳ chọn) Baseline chưa fine-tune, để so sánh

In [ ]:
for split in ["dev_seen", "dev_unseen"]:
    !python agent.py --ids splits/{split}.txt --out runs/base_{split}.jsonl 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|Error|Traceback"
    !python score.py runs/base_{split}.jsonl